# STORM-like Random Energy Simulator — No Machine Learning

This notebook builds the first stage of the PhD mini-project. It generates consumers, weather, a radial electrical network, physical diagnostics, STORM-compatible data, anomaly labels, and diagrams. All behaviour is rule-based or random; no ML is used.

## 1. Imports

If a package is missing, run `pip install -r requirements.txt` in a terminal.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import json

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Image, display

from energy_simulator import (
    ConsumerFactory, ContextGenerator, RadialGrid,
    RuleBasedDemandModel, SimulationConfig,
    StormLikePublisher, draw_plots,
)

pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")

## 2. Set every simulation parameter

Change these values to create a different experiment. The random seed makes the result reproducible.

In [ ]:
config = SimulationConfig(
    seed=42,
    days=7,
    interval_minutes=15,
    n_consumers=12,
    start="2026-01-01 00:00:00+00:00",
    nominal_voltage_v=11_000.0,
    power_factor=0.95,
    line_resistance_ohm_per_km=0.12,
    min_line_capacity_kw=80.0,
    max_line_capacity_kw=500.0,
    minimum_allowed_voltage_pu=0.90,
    maximum_allowed_voltage_pu=1.10,
    measurement_noise_std_fraction=0.012,
    bottom_up_noise_std_fraction=0.035,
    anomaly_probability=0.003,
    switch_event_probability_per_day=0.05,
    missing_probability=0.003,
    uncertain_boundary_steps=2,
)
config.validate()
rng = np.random.default_rng(config.seed)
OUTPUT = Path("notebook_results")
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.Series(asdict(config), name="value").to_frame()

## 3. Create continuous timestamps and random weather

The output uses the same 15-minute frequency as STORM. Weather is synthetic and combines daily cycles with random variation.

In [ ]:
n_steps = config.days * 1440 // config.interval_minutes
timestamps = pd.date_range(
    config.start, periods=n_steps, freq=f"{config.interval_minutes}min"
)
context = ContextGenerator(rng).generate(timestamps)
print("Rows:", len(context))
print("Time interval:", context["M_TIMESTAMP"].diff().dropna().mode().iloc[0])
display(context.head())

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(context["M_TIMESTAMP"], context["temperature_c"])
axes[0].set_ylabel("Temperature (°C)")
axes[1].plot(context["M_TIMESTAMP"], context["solar_irradiance_w_m2"], color="darkorange")
axes[1].set_ylabel("Irradiance (W/m²)")
axes[1].set_xlabel("UTC time")
fig.suptitle("Random simulation context")
plt.tight_layout()
plt.show()

## 4. Randomly create consumers

The factory produces homes, commercial buildings, and industrial consumers with private characteristics such as household size, floor area, PV, and EV ownership.

In [ ]:
consumers = [
    ConsumerFactory.random(f"C{i:03d}", rng)
    for i in range(1, config.n_consumers + 1)
]
consumer_metadata = pd.DataFrame(asdict(consumer) for consumer in consumers)
display(consumer_metadata)
display(consumer_metadata["consumer_type"].value_counts().to_frame("count"))

## 5. Generate consumption, PV, EV, and net-load time series

The rule-based model responds to consumer type, opening hours, weekday/weekend, temperature, solar irradiance, and random variation.

In [ ]:
demand_model = RuleBasedDemandModel(rng)
consumer_timeseries = pd.concat(
    [demand_model.calculate(consumer, context) for consumer in consumers],
    ignore_index=True,
)
display(consumer_timeseries.head())

example_id = consumers[0].consumer_id
example = consumer_timeseries[consumer_timeseries["consumer_id"] == example_id]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(example["M_TIMESTAMP"], example["consumption_kw"], label="Consumption")
ax.plot(example["M_TIMESTAMP"], example["pv_generation_kw"], label="PV generation")
ax.plot(example["M_TIMESTAMP"], example["net_load_kw"], label="Net load")
ax.set(title=f"Example consumer {example_id}", xlabel="UTC time", ylabel="Power (kW)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Construct a random radial electrical network

The substation is the source. Each edge represents a line with distance, capacity, and resistance.

In [ ]:
grid = RadialGrid.random(consumers, config, rng)
network_edges = pd.DataFrame([
    {**asdict(line), "resistance_ohm": line.resistance_ohm}
    for line in grid.lines
])
display(network_edges)

graph = nx.DiGraph()
graph.add_node("SUBSTATION", kind="substation")
for consumer in consumers:
    graph.add_node(consumer.consumer_id, kind=consumer.consumer_type)
graph.add_edges_from((line.parent, line.child) for line in grid.lines)
colors = {"substation": "#d62728", "home": "#1f77b4", "commercial": "#ff7f0e", "industrial": "#2ca02c"}
pos = nx.spring_layout(graph, seed=11)
plt.figure(figsize=(12, 8))
nx.draw_networkx(
    graph, pos, node_color=[colors[graph.nodes[n]["kind"]] for n in graph],
    node_size=900, arrows=True, font_size=8,
)
nx.draw_networkx_edge_labels(
    graph, pos,
    edge_labels={(line.parent, line.child): f"{line.distance_km:.2f} km\n{line.capacity_kw:.0f} kW" for line in grid.lines},
    font_size=6,
)
plt.title("Synthetic radial distribution network")
plt.axis("off")
plt.show()

## 7. Calculate approximate physical quantities

The model calculates downstream power, three-phase current, resistive loss, approximate voltage drop, line loading, overload flags, and undervoltage flags. This is an educational approximation rather than an AC power-flow solver.

In [ ]:
line_timeseries = grid.evaluate(consumer_timeseries)
physical_summary = pd.Series({
    "maximum_line_loading_percent": line_timeseries["loading_percent"].max(),
    "minimum_voltage_pu": line_timeseries["voltage_pu"].min(),
    "overloaded_line_rows": line_timeseries["overloaded"].sum(),
    "voltage_violation_rows": line_timeseries["voltage_violation"].sum(),
    "total_energy_loss_kwh": line_timeseries["loss_kw"].sum() * config.interval_minutes / 60,
})
display(physical_summary.to_frame("value"))
display(line_timeseries.head())

## 8. Create STORM-compatible measurements and labels

`S_original` represents a noisy substation measurement. `BU_original` represents a noisy bottom-up estimate. Labels are 0 for normal operation, 1 for an event, and 5 for an uncertain event boundary.

In [ ]:
x_data, y_data = StormLikePublisher(config, rng).create(consumer_timeseries)
storm_data = x_data.merge(y_data, on="M_TIMESTAMP", validate="one_to_one")
display(storm_data.head())
display(storm_data["label"].value_counts().sort_index().to_frame("count"))

assert len(storm_data) == n_steps
assert storm_data["M_TIMESTAMP"].diff().dropna().eq(pd.Timedelta(minutes=config.interval_minutes)).all()
assert set(storm_data["label"].unique()).issubset({0, 1, 5})
print("Validation passed.")

## 9. Draw and save all project diagrams

In [ ]:
draw_plots(OUTPUT, consumers, grid, x_data, y_data, line_timeseries)
for filename in [
    "storm_timeseries.png", "residual_and_events.png",
    "daily_profile.png", "physical_diagnostics.png",
]:
    print(filename)
    display(Image(filename=str(OUTPUT / "plots" / filename), width=1000))

## 10. Save the dataset and all parameters

The private consumer metadata is deliberately stored separately from the STORM-style aggregate data.

In [ ]:
x_dir = OUTPUT / "Train" / "X"
y_dir = OUTPUT / "Train" / "y"
x_dir.mkdir(parents=True, exist_ok=True)
y_dir.mkdir(parents=True, exist_ok=True)

x_data.to_csv(x_dir / "1.csv", index=False)
y_data.to_csv(y_dir / "1.csv", index=False)
consumer_metadata.to_csv(OUTPUT / "consumer_metadata.csv", index=False)
consumer_timeseries.to_csv(OUTPUT / "consumer_timeseries.csv", index=False)
context.to_csv(OUTPUT / "context_timeseries.csv", index=False)
network_edges.to_csv(OUTPUT / "network_edges.csv", index=False)
line_timeseries.to_csv(OUTPUT / "line_timeseries.csv", index=False)
(OUTPUT / "simulation_config.json").write_text(
    json.dumps(asdict(config), indent=2), encoding="utf-8"
)

summary = {
    "timestamps": len(x_data),
    "consumers": len(consumers),
    "normal_rows": int((y_data["label"] == 0).sum()),
    "event_rows": int((y_data["label"] == 1).sum()),
    "uncertain_rows": int((y_data["label"] == 5).sum()),
    "missing_rows": int(x_data["missing"].sum()),
    **{key: float(value) for key, value in physical_summary.items()},
}
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(pd.Series(summary, name="value").to_frame())
print("Saved in:", OUTPUT.resolve())

## Next stage: machine learning

The next notebook can transform each normal day into a 96-value vector and train a conditional VAE. It should compare statistical fidelity and physical violation rates against this non-ML baseline.